# Cleaning and Feature Engineering in Dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('smart_hospital_appointment_dataset.csv')

In [3]:
#copied data as df 
df = data.copy()

In [5]:
# Check missing values in chronic_disease as we have seen while understanding dataset = 4450
df["chronic_disease"].isnull().sum()

np.int64(4450)

In [6]:
# we will replace null values Nan in chronic disease columns i.e. 4450 total with 'No chronic disease'
df["chronic_disease"] = df["chronic_disease"].fillna("No Chronic Disease")

In [7]:
#count of different values in column chronic_disease 
df["chronic_disease"].value_counts()

chronic_disease
No Chronic Disease    4450
Diabetes              2034
Hypertension          2018
Asthma                1018
Heart Disease          480
Name: count, dtype: int64

now we can see the no chronic disease value is 4450 

In [16]:
#To analyze the dataset for possible outliners if any uncertiain  data is found

numerical_columns = df.select_dtypes(include=['int64','float64']).columns #data as column

#use to change columns such as age , bmi with count , mean for better readability

df[numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
age,10000.0,45.00040,25.862836,1.0,22.00,45.00,68.0,89.0
bmi,10000.0,24.01089,3.935325,15.0,21.30,24.00,26.7,39.8
doctor_experience_years,10000.0,15.39900,8.604069,1.0,8.00,15.00,23.0,30.0
consultation_fee,10000.0,606.86070,180.246694,220.0,461.00,609.00,752.0,999.0
booking_to_appointment_days,10000.0,12.75450,9.405715,0.0,4.00,12.00,21.0,29.0
waiting_time_minutes,10000.0,25.20390,14.862961,1.0,13.00,23.00,35.0,59.0
previous_appointments,10000.0,7.01920,4.334502,0.0,3.00,7.00,11.0,14.0
missed_previous_appointments,10000.0,1.41670,1.383637,0.0,0.00,1.00,2.0,9.0
hospital_rating,10000.0,3.98994,0.579424,3.0,3.50,4.00,4.5,5.0
emergency_case,10000.0,0.14860,0.355712,0.0,0.00,0.00,0.0,1.0


As we can clearly observe that there is not a single unrealistic numeric values in dataset summary. So, no outliner removal step is performed 


In [17]:
df["age"].describe()

count    10000.000000
mean        45.000400
std         25.862836
min          1.000000
25%         22.000000
50%         45.000000
75%         68.000000
max         89.000000
Name: age, dtype: float64

In [19]:
# feature Engineering age group 
age_bins = [0, 12, 17, 25, 55, 89]

age_labels = [
    "Child",
    "Teen",
    "Young Adult",
    "Middle Aged",
    "Senior"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

In [22]:
df["age_group"].value_counts()

age_group
Senior         3837
Middle Aged    3293
Child          1361
Young Adult     915
Teen            594
Name: count, dtype: int64

In [24]:
#checking random 10 samples
df[["age","age_group"]].sample(10)

,age,age_group
1571,64,Senior
331,48,Middle Aged
2104,58,Senior
4662,84,Senior
6424,78,Senior
1304,31,Middle Aged
4172,19,Young Adult
8796,48,Middle Aged
9735,73,Senior
5802,5,Child


In [26]:
df["bmi"].describe()

count    10000.000000
mean        24.010890
std          3.935325
min         15.000000
25%         21.300000
50%         24.000000
75%         26.700000
max         39.800000
Name: bmi, dtype: float64

In [31]:
# feature engineering bmi of patients into different categories 
bmi_bins = [0, 18.5, 25, 30, float("inf")]

bmi_labels = [
    "Underweight",
    "Normal",
    "Overweight",
    "Obese"
]

df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=bmi_bins,
    labels=bmi_labels,
    include_lowest=True
)

In [32]:
df["bmi_category"].value_counts()

bmi_category
Normal         5213
Overweight     3332
Underweight     843
Obese           612
Name: count, dtype: int64

In [33]:
df[["age_group","bmi_category"]].sample(10)

,age_group,bmi_category
9534,Senior,Normal
3429,Young Adult,Overweight
613,Child,Overweight
5426,Middle Aged,Normal
4666,Middle Aged,Underweight
5681,Teen,Overweight
8363,Senior,Overweight
5015,Young Adult,Overweight
8508,Senior,Overweight
6417,Senior,Overweight


Feature Engineering both bmi_category and Age_group was very important as categorical data is more beneficial rather than actual value .
Analysis based on category became more easy and informative .

In [35]:
#feature engineering first time patient and pervious patient 
df["repeat_patient"] = np.where(
    df["previous_appointments"] > 0,
    "Yes",
    "No"
)

In [36]:
df[["patient_id","repeat_patient"]].sample(10)

,patient_id,repeat_patient
4818,PAT_14818,Yes
87,PAT_10087,Yes
2500,PAT_12500,Yes
6671,PAT_16671,Yes
7068,PAT_17068,Yes
7409,PAT_17409,Yes
6328,PAT_16328,Yes
9804,PAT_19804,Yes
9730,PAT_19730,Yes
6114,PAT_16114,Yes


In [38]:
df["repeat_patient"].value_counts()

repeat_patient
Yes    9312
No      688
Name: count, dtype: int64

Feature engineering for patient_repetiton help us to compare satisfaction , waiting time , billing experience and satisfaction with a new customer 

In [40]:
df["waiting_time_minutes"].describe()

count    10000.000000
mean        25.203900
std         14.862961
min          1.000000
25%         13.000000
50%         23.000000
75%         35.000000
max         59.000000
Name: waiting_time_minutes, dtype: float64

In [41]:
#feature engineering longwait flags
df["long_wait_flag"] = np.where(
    df["waiting_time_minutes"] > 30,
    "Yes",
    "No"
)

In [42]:
df[["patient_id","long_wait_flag"]].sample(10)

,patient_id,long_wait_flag
7727,PAT_17727,Yes
7363,PAT_17363,No
9448,PAT_19448,Yes
2290,PAT_12290,No
8203,PAT_18203,Yes
8434,PAT_18434,Yes
7156,PAT_17156,No
9362,PAT_19362,Yes
7289,PAT_17289,Yes
8369,PAT_18369,No


In [43]:
df["long_wait_flag"].value_counts()

long_wait_flag
No     6446
Yes    3554
Name: count, dtype: int64

Patients waiting more than 30 minutes are categorized as having experienced a long wait. This feature help the analysis of patient satisfaction, appointment outcomes, and department performance.

In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 30 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   patient_id                    10000 non-null  str     
 1   age                           10000 non-null  int64   
 2   gender                        10000 non-null  str     
 3   city                          10000 non-null  str     
 4   bmi                           10000 non-null  float64 
 5   chronic_disease               10000 non-null  str     
 6   appointment_type              10000 non-null  str     
 7   department                    10000 non-null  str     
 8   symptoms                      10000 non-null  str     
 9   severity_level                10000 non-null  str     
 10  doctor_experience_years       10000 non-null  int64   
 11  consultation_fee              10000 non-null  int64   
 12  insurance                     10000 non-null  str     
 13

In [45]:
df.isnull().sum()

patient_id                      0
age                             0
gender                          0
city                            0
bmi                             0
chronic_disease                 0
appointment_type                0
department                      0
symptoms                        0
severity_level                  0
doctor_experience_years         0
consultation_fee                0
insurance                       0
appointment_day                 0
appointment_month               0
booking_to_appointment_days     0
waiting_time_minutes            0
previous_appointments           0
missed_previous_appointments    0
hospital_rating                 0
emergency_case                  0
patient_satisfaction_score      0
medicine_cost                   0
test_cost                       0
total_bill                      0
appointment_status              0
age_group                       0
bmi_category                    0
repeat_patient                  0
long_wait_flag

In [46]:
df.to_csv(
    "healthcare_appointments_cleaned.csv",
    index=False
)

# 4 rows are added to dataset such as Age_group, long_wait_flag, repeat_patient, bmi_category